In [234]:
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
import platform
import matplotlib.pyplot as plt

# 출력 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

In [235]:
portfolio = pd.read_csv('../../../integrated/data/portfolio.csv')
profile = pd.read_csv('../../../integrated/data/profile.csv')
transcript = pd.read_csv('../../../integrated/data/transcript.csv')
menu = pd.read_csv('../../../integrated/data/starbucks_menu_260112.csv')

# 분석 기준
- 전체 전환율 먼저 확인
- 전환 기준: offer received 이후 일정 시간 내 transaction
- 고객 세그먼트: 구매횟수, 구매금액 분위수 기반
- VIP: 고활동 + 고금액, 일반 고객(전환율보고 판단)
- 이후 프로모션별, 채널별 비교

# 전환율 기준
- 기준: offer received(프로모션 받음)
- 전환: 일정 시간 내 transaction(거래)
- 시간: 72시간으로 이내로 정함(time) 3일로 단기 프로모션 느낌으로 반응확인

- offer received 추출
- transaction 추출
- 시간 차이 계산
- 72시간 내 여부 → converted
- 전체 전환율 계산

전처리팀 끝나는데로 시작

일단 원본데이터로 계산

In [236]:
transcript['event'].value_counts()

event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33579
Name: count, dtype: int64

In [237]:
transcript.head()

,Unnamed: 0,person,event,value,time
0,0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0
3,3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0
4,4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0


In [238]:
offer = transcript[transcript['event'] == 'offer received'].copy()
transaction = transcript[transcript['event'] == 'transaction'].copy()

In [239]:
offer

,Unnamed: 0,person,event,value,time
0,0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0
3,3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0
4,4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0
...,...,...,...,...,...
257882,257882,d087c473b4d247ccb0abfef59ba12b0e,offer received,{'offer id': 'ae264e3637204a6fb9bb56bc8210ddfd'},576
257883,257883,cb23b66c56f64b109d673d5e56574529,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},576
257884,257884,6d5f3a774f3d4714ab0c092238f3a1d7,offer received,{'offer id': '2298d6c36e964ae4a3e7e9706d1fb8c2'},576
257885,257885,9dc1421481194dcd9400aec7c9ae6366,offer received,{'offer id': 'ae264e3637204a6fb9bb56bc8210ddfd'},576


In [240]:
offer.isna().sum()

Unnamed: 0    0
person        0
event         0
value         0
time          0
dtype: int64

In [241]:
offer.shape

(76277, 5)

In [242]:
transaction.isna().sum()

Unnamed: 0    0
person        0
event         0
value         0
time          0
dtype: int64

In [243]:
transaction

,Unnamed: 0,person,event,value,time
12654,12654,02c083884c7d45b39cc68e1314fec56c,transaction,{'amount': 0.8300000000000001},0
12657,12657,9fa9ae8f57894cc9a3b8a9bbe0fc1b2f,transaction,{'amount': 34.56},0
12659,12659,54890f68699049c2a04d415abc25e717,transaction,{'amount': 13.23},0
12670,12670,b2f1cd155b864803ad8334cdf13c4bd2,transaction,{'amount': 19.51},0
12671,12671,fe97aa22dd3e48c8b143116a8403dd52,transaction,{'amount': 18.97},0
...,...,...,...,...,...
306529,306529,b3a1272bc9904337b331bf348c3e8c17,transaction,{'amount': 1.5899999999999999},714
306530,306530,68213b08d99a4ae1b0dcb72aebd9aa35,transaction,{'amount': 9.53},714
306531,306531,a00058cf10334a308c68e7631c529907,transaction,{'amount': 3.61},714
306532,306532,76ddbd6576844afe811f1a3c0fbb5bec,transaction,{'amount': 3.5300000000000002},714


In [244]:
transaction.shape

(138953, 5)

In [245]:
transaction['value'].head()

12654    {'amount': 0.8300000000000001}
12657                 {'amount': 34.56}
12659                 {'amount': 13.23}
12670                 {'amount': 19.51}
12671                 {'amount': 18.97}
Name: value, dtype: str

In [246]:
import ast
transaction['value'] = transaction['value'].apply(ast.literal_eval)

In [247]:
type(transaction['value'].iloc[0])

dict

In [248]:
transaction['value'].head()

12654    {'amount': 0.8300000000000001}
12657                 {'amount': 34.56}
12659                 {'amount': 13.23}
12670                 {'amount': 19.51}
12671                 {'amount': 18.97}
Name: value, dtype: object

In [249]:
import ast

offer['offer_id'] = offer['value'].apply(
    lambda x: ast.literal_eval(x)['offer id'] if isinstance(x, str) else x['offer id']
)

transaction['amount'] = transaction['value'].apply(
    lambda x: ast.literal_eval(x)['amount'] if isinstance(x, str) else x['amount']
)

In [250]:
offer

,Unnamed: 0,person,event,value,time,offer_id
0,0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0,9b98b8c7a33c4b65b9aebfe6a799e6d9
1,1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0,0b1e1539f2cc45b7b9fa7c272da2e1d7
2,2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0,2906b810c7d4411798c6938adc9daaa5
3,3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0,fafdcd668e3743c1bb461111dcafc2a4
4,4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0,4d5c57ea9a6940dd891ad53e9dbe8da0
...,...,...,...,...,...,...
257882,257882,d087c473b4d247ccb0abfef59ba12b0e,offer received,{'offer id': 'ae264e3637204a6fb9bb56bc8210ddfd'},576,ae264e3637204a6fb9bb56bc8210ddfd
257883,257883,cb23b66c56f64b109d673d5e56574529,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},576,2906b810c7d4411798c6938adc9daaa5
257884,257884,6d5f3a774f3d4714ab0c092238f3a1d7,offer received,{'offer id': '2298d6c36e964ae4a3e7e9706d1fb8c2'},576,2298d6c36e964ae4a3e7e9706d1fb8c2
257885,257885,9dc1421481194dcd9400aec7c9ae6366,offer received,{'offer id': 'ae264e3637204a6fb9bb56bc8210ddfd'},576,ae264e3637204a6fb9bb56bc8210ddfd


In [251]:
transaction

,Unnamed: 0,person,event,value,time,amount
12654,12654,02c083884c7d45b39cc68e1314fec56c,transaction,{'amount': 0.8300000000000001},0,0.83
12657,12657,9fa9ae8f57894cc9a3b8a9bbe0fc1b2f,transaction,{'amount': 34.56},0,34.56
12659,12659,54890f68699049c2a04d415abc25e717,transaction,{'amount': 13.23},0,13.23
12670,12670,b2f1cd155b864803ad8334cdf13c4bd2,transaction,{'amount': 19.51},0,19.51
12671,12671,fe97aa22dd3e48c8b143116a8403dd52,transaction,{'amount': 18.97},0,18.97
...,...,...,...,...,...,...
306529,306529,b3a1272bc9904337b331bf348c3e8c17,transaction,{'amount': 1.5899999999999999},714,1.59
306530,306530,68213b08d99a4ae1b0dcb72aebd9aa35,transaction,{'amount': 9.53},714,9.53
306531,306531,a00058cf10334a308c68e7631c529907,transaction,{'amount': 3.61},714,3.61
306532,306532,76ddbd6576844afe811f1a3c0fbb5bec,transaction,{'amount': 3.5300000000000002},714,3.53


In [252]:
# 필요한 컬럼만 정리
offer = offer[['person', 'time', 'offer_id']].rename(columns={'time': 'offer_time'})
transaction = transaction[['person', 'time', 'amount']].rename(columns={'time': 'trans_time'})

In [253]:
# 같은 고객 기준으로 offer와 transaction 연결
conv = offer.merge(transaction, on='person', how='left')

In [254]:
# offer 이후 몇 시간 뒤 구매했는지 계산
conv['time_diff'] = conv['trans_time'] - conv['offer_time']

In [255]:
# 72시간 이내 구매면 전환
conv['converted'] = ((conv['time_diff'] >= 0) & (conv['time_diff'] <= 72)).astype(int)

In [256]:
conv_final = conv.groupby(['person', 'offer_time', 'offer_id'], as_index=False)['converted'].max()

In [257]:
overall_conversion_rate = conv_final['converted'].mean()
converted_cnt = conv_final['converted'].sum()
non_converted_cnt = len(conv_final) - converted_cnt

print('전체 offer 수:', len(conv_final))
print('전환 수:', converted_cnt)
print('비전환 수:', non_converted_cnt)
print('전체 전환율:', round(overall_conversion_rate * 100, 2), '%')
print('전체 비전환율:', round((1 - overall_conversion_rate) * 100, 2), '%')

전체 offer 수: 76277
전환 수: 46008
비전환 수: 30269
전체 전환율: 60.32 %
전체 비전환율: 39.68 %


In [258]:
# 1은 전환 고객 0은 비전환 고객
conv_final['converted'].value_counts()

converted
1    46008
0    30269
Name: count, dtype: int64

In [259]:
df = pd.read_csv('../../../integrated/data/transcript_portfolio_profile.csv')

In [260]:
df.head()

,customer_id,event,time,offer_id,amount,event_reward,offer_type,offer_reward,difficulty,duration,channels,web,email,mobile,social,gender,age,income,became_member_on,value
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,bogo,5.0,5.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,F,75.0,100000.0,2017-05-09,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'}
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,discount,5.0,20.0,10.0,"['web', 'email']",1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,discount,2.0,10.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,M,68.0,70000.0,2018-04-26,{'offer id': '2906b810c7d4411798c6938adc9daaa5'}
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,discount,2.0,10.0,10.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'}
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,bogo,10.0,10.0,5.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'}


1차 전처리한 데이터로 다시 전체 전환율 및 비전환율 계산

In [261]:
df2 = df.copy()

In [262]:
df2['event'].value_counts()

event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33182
Name: count, dtype: int64

- 일단 event컬럼에서 프로모션을 받은 컬럼이랑 거래컬럼 추출
- 프로모션을 받은 고객들대상으로 일정시간 후 구매여부 확인
- 프로모션으로 인해 구매했는지 보기위해 시간을 72시간으로 제한

In [263]:
df2['value'].head()

0    {'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'}
1    {'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}
2    {'offer id': '2906b810c7d4411798c6938adc9daaa5'}
3    {'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'}
4    {'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'}
Name: value, dtype: str

In [264]:
# 혹시몰라 value_dict로 파생컬럼생성 후 dict로 변환
df2['value_dict'] = df2['value'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [265]:
df2['value_dict'].head()

0    {'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'}
1    {'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}
2    {'offer id': '2906b810c7d4411798c6938adc9daaa5'}
3    {'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'}
4    {'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'}
Name: value_dict, dtype: object

In [266]:
offer_clean = df2[df2['event'] == 'offer received'].copy()
transaction_clean = df2[df2['event'] == 'transaction'].copy()

In [267]:
offer_clean.head()

,customer_id,event,time,offer_id,amount,event_reward,offer_type,offer_reward,difficulty,duration,channels,web,email,mobile,social,gender,age,income,became_member_on,value,value_dict
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,bogo,5.0,5.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,F,75.0,100000.0,2017-05-09,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'}
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,discount,5.0,20.0,10.0,"['web', 'email']",1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,discount,2.0,10.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,M,68.0,70000.0,2018-04-26,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},{'offer id': '2906b810c7d4411798c6938adc9daaa5'}
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,discount,2.0,10.0,10.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'}
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,bogo,10.0,10.0,5.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'}


In [268]:
transaction_clean

,customer_id,event,time,offer_id,amount,event_reward,offer_type,offer_reward,difficulty,duration,channels,web,email,mobile,social,gender,age,income,became_member_on,value,value_dict
12654,02c083884c7d45b39cc68e1314fec56c,transaction,0,NaN,0.83,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,F,20.0,30000.0,2016-07-11,{'amount': 0.8300000000000001},{'amount': 0.8300000000000001}
12657,9fa9ae8f57894cc9a3b8a9bbe0fc1b2f,transaction,0,NaN,34.56,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,M,42.0,96000.0,2016-01-17,{'amount': 34.56},{'amount': 34.56}
12659,54890f68699049c2a04d415abc25e717,transaction,0,NaN,13.23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,M,36.0,56000.0,2017-12-28,{'amount': 13.23},{'amount': 13.23}
12670,b2f1cd155b864803ad8334cdf13c4bd2,transaction,0,NaN,19.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,F,55.0,94000.0,2017-10-16,{'amount': 19.51},{'amount': 19.51}
12671,fe97aa22dd3e48c8b143116a8403dd52,transaction,0,NaN,18.97,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,F,39.0,67000.0,2017-12-17,{'amount': 18.97},{'amount': 18.97}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306132,b3a1272bc9904337b331bf348c3e8c17,transaction,714,NaN,1.59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,M,66.0,47000.0,2018-01-01,{'amount': 1.5899999999999999},{'amount': 1.5899999999999999}
306133,68213b08d99a4ae1b0dcb72aebd9aa35,transaction,714,NaN,9.53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,M,52.0,62000.0,2018-04-08,{'amount': 9.53},{'amount': 9.53}
306134,a00058cf10334a308c68e7631c529907,transaction,714,NaN,3.61,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,F,63.0,52000.0,2013-09-22,{'amount': 3.61},{'amount': 3.61}
306135,76ddbd6576844afe811f1a3c0fbb5bec,transaction,714,NaN,3.53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,M,57.0,40000.0,2016-07-09,{'amount': 3.5300000000000002},{'amount': 3.5300000000000002}


In [227]:
offer_clean.isna().sum()

customer_id             0
event                   0
time                    0
offer_id                0
amount              76277
event_reward        76277
offer_type              0
offer_reward            0
difficulty              0
duration                0
channels                0
web                     0
email                   0
mobile                  0
social                  0
gender               9776
age                  9776
income               9776
became_member_on     9776
value                   0
value_dict              0
dtype: int64

In [228]:
transaction_clean.isna().sum()

customer_id              0
event                    0
time                     0
offer_id            138953
amount                   0
event_reward        138953
offer_type          138953
offer_reward        138953
difficulty          138953
duration            138953
channels            138953
web                 138953
email               138953
mobile              138953
social              138953
gender               14996
age                  14996
income               14996
became_member_on     14996
value                    0
value_dict               0
dtype: int64

파생컬럼 만들었다 결측이 생겨 필요컬럼만 쓰고 나머지는 버리는 쪽으로....

In [229]:
offer_clean = offer_clean[['customer_id', 'time', 'offer_id']]
transaction_clean = transaction_clean[['customer_id', 'time', 'amount']]

In [230]:
offer_clean

,customer_id,time,offer_id
0,78afa995795e4d85b5d9ceeca43f5fef,0,9b98b8c7a33c4b65b9aebfe6a799e6d9
1,a03223e636434f42ac4c3df47e8bac43,0,0b1e1539f2cc45b7b9fa7c272da2e1d7
2,e2127556f4f64592b11af22de27a7932,0,2906b810c7d4411798c6938adc9daaa5
3,8ec6ce2a7e7949b1bf142def7d0e0586,0,fafdcd668e3743c1bb461111dcafc2a4
4,68617ca6246f4fbc85e91a2a49552598,0,4d5c57ea9a6940dd891ad53e9dbe8da0
...,...,...,...
257635,d087c473b4d247ccb0abfef59ba12b0e,576,ae264e3637204a6fb9bb56bc8210ddfd
257636,cb23b66c56f64b109d673d5e56574529,576,2906b810c7d4411798c6938adc9daaa5
257637,6d5f3a774f3d4714ab0c092238f3a1d7,576,2298d6c36e964ae4a3e7e9706d1fb8c2
257638,9dc1421481194dcd9400aec7c9ae6366,576,ae264e3637204a6fb9bb56bc8210ddfd


In [231]:
transaction_clean

,customer_id,time,amount
12654,02c083884c7d45b39cc68e1314fec56c,0,0.83
12657,9fa9ae8f57894cc9a3b8a9bbe0fc1b2f,0,34.56
12659,54890f68699049c2a04d415abc25e717,0,13.23
12670,b2f1cd155b864803ad8334cdf13c4bd2,0,19.51
12671,fe97aa22dd3e48c8b143116a8403dd52,0,18.97
...,...,...,...
306132,b3a1272bc9904337b331bf348c3e8c17,714,1.59
306133,68213b08d99a4ae1b0dcb72aebd9aa35,714,9.53
306134,a00058cf10334a308c68e7631c529907,714,3.61
306135,76ddbd6576844afe811f1a3c0fbb5bec,714,3.53


In [269]:
# 시간차이 계산 프로모션받고 몇시간뒤에 구매했는지
conv['time_diff'] = conv['trans_time'] - conv['offer_time']

In [270]:
conv

,person,offer_time,offer_id,trans_time,amount,time_diff,converted
0,78afa995795e4d85b5d9ceeca43f5fef,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,132.0,19.89,132.0,0
1,78afa995795e4d85b5d9ceeca43f5fef,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,144.0,17.78,144.0,0
2,78afa995795e4d85b5d9ceeca43f5fef,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,222.0,19.67,222.0,0
3,78afa995795e4d85b5d9ceeca43f5fef,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,240.0,29.72,240.0,0
4,78afa995795e4d85b5d9ceeca43f5fef,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,378.0,23.93,378.0,0
...,...,...,...,...,...,...,...
640335,e4052622e5ba45a8b96b59aba68cf068,576,3f207df678b143eea3cee63160fa8bed,84.0,25.19,-492.0,0
640336,e4052622e5ba45a8b96b59aba68cf068,576,3f207df678b143eea3cee63160fa8bed,96.0,21.53,-480.0,0
640337,e4052622e5ba45a8b96b59aba68cf068,576,3f207df678b143eea3cee63160fa8bed,480.0,30.57,-96.0,0
640338,e4052622e5ba45a8b96b59aba68cf068,576,3f207df678b143eea3cee63160fa8bed,486.0,19.47,-90.0,0


In [ ]:
# 음수값이 생겨 프로모션 받기전 데이터는 제거
conv = conv[conv['time_diff'] >= 0]

In [ ]:
# 프로모션을 받고 구매한 가장 가까운데이터만 남기기
conv = conv.sort_values('time_diff').drop_duplicates(
    ['person', 'offer_time', 'offer_id']
)

In [ ]:
# 72시간 기준 전환율
conv['converted'] = (conv['time_diff'] <= 72).astype(int)

In [ ]:
conversion_rate = conv['converted'].mean()

print('전체 전환율:', round(conversion_rate * 100, 2),'%')

전체 전환율: 85.05 %


프로모션 이후 발생한 거래 중 가장 빠른 구매를 기준으로 고객 반응을 정의하여 노이즈를 줄이고 전환 분석의 정확도높임
그렇게 보니 전체 전환율 85.05% 나옴

결론: 전체 프로모션 수신 건 중 약 85.05%가 72시간 이내 실제 구매로 이어짐

그래서 프로모션별 전환율도 확인해봐야될듯

In [ ]:
conv

,person,offer_time,offer_id,trans_time,amount,time_diff,converted
603592,4004ab48104f402aa580c53b4d09d9dd,576,4d5c57ea9a6940dd891ad53e9dbe8da0,0.0,2.40,-576.0,1
551932,917c006586784776a1d11ba0e7671158,576,4d5c57ea9a6940dd891ad53e9dbe8da0,0.0,2.04,-576.0,1
597609,581ad5ee9b9f438a8856a94b9710ec45,576,0b1e1539f2cc45b7b9fa7c272da2e1d7,0.0,16.87,-576.0,1
580248,a05d5de8d4314ec19162578f67af3fee,576,2298d6c36e964ae4a3e7e9706d1fb8c2,0.0,16.30,-576.0,1
551652,a442df3deaaf4296b5c25564422dc9c7,576,f19421c1d4aa40978ebb69ca19b0e20d,0.0,2.49,-576.0,1
...,...,...,...,...,...,...,...
637504,4056d15bfaf94aca9600d139493cee45,576,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,NaN,0
637834,0f4eeccda01547bdaa30113a09725a2b,576,3f207df678b143eea3cee63160fa8bed,NaN,NaN,NaN,0
638236,4c9c818067914caba08ca3cebadab165,576,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,NaN,0
638661,1f5e62e325d047f4833d1b7aed437f31,576,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,NaN,0


In [ ]:
conv.groupby('offer_id')['converted'].mean()

offer_id
0b1e1539f2cc45b7b9fa7c272da2e1d7    0.816641
2298d6c36e964ae4a3e7e9706d1fb8c2    0.873921
2906b810c7d4411798c6938adc9daaa5    0.830189
3f207df678b143eea3cee63160fa8bed    0.827491
4d5c57ea9a6940dd891ad53e9dbe8da0    0.872909
5a8bc65990b245e5a138643cd4eb9837    0.859018
9b98b8c7a33c4b65b9aebfe6a799e6d9    0.834180
ae264e3637204a6fb9bb56bc8210ddfd    0.853487
f19421c1d4aa40978ebb69ca19b0e20d    0.866860
fafdcd668e3743c1bb461111dcafc2a4    0.870607
Name: converted, dtype: float64

In [ ]:
conv.groupby('offer_id')['converted'].agg(['mean', 'count'])

,mean,count
offer_id,,
0b1e1539f2cc45b7b9fa7c272da2e1d7,0.816641,7668
2298d6c36e964ae4a3e7e9706d1fb8c2,0.873921,7646
2906b810c7d4411798c6938adc9daaa5,0.830189,7632
3f207df678b143eea3cee63160fa8bed,0.827491,7617
4d5c57ea9a6940dd891ad53e9dbe8da0,0.872909,7593
5a8bc65990b245e5a138643cd4eb9837,0.859018,7618
9b98b8c7a33c4b65b9aebfe6a799e6d9,0.834180,7677
ae264e3637204a6fb9bb56bc8210ddfd,0.853487,7658
f19421c1d4aa40978ebb69ca19b0e20d,0.866860,7571


아이디 기준 해당 프로모션을 받았을때 평균 81.6퍼가 구매로 이어짐 그리고 프로모션을 7668번 받았다

현재 85.05%가 이 오퍼때문에 구매했다가 아니라 이 고객이 나중에 한 번이라도 샀다에 가까운 값이 되어서 다시 구해야봐야 됨...